# Commission Data Merge
**Outputs stripped from repository version.**

Merges WI activations log, TMO activations statement, and TMO activation detail for PT#s. Draws on preprocessed data inputs.

In [1]:
from load_wi_modules import *

Core modules loaded.
WI modules loaded.


In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
from check_version import checkVersion

Envio - bq-env
Python - 3.11.8


In [ ]:
wi.Month.value_counts()

In [ ]:
tmobile.Month.value_counts()

### Prep

In [ ]:
from update_ban_list import *   
banList.Rep.value_counts().head()
banList.Rep.value_counts().sort_index()

In [ ]:
## Residual processing
from res_comp_calc import resComp

#- Exports breakdown for Thomas.
from residual_pivot import resPivot
resPivot.shape

assert ~residual.duplicated().any(), 'Duplicated residuals'
residual.shape

In [ ]:
fallout.shape

# Merge

In [ ]:
#tmobile.dtypes

In [ ]:
#- roll tmo
gCols = ['PTNumber', 'BAN', 'ActivityType', 'Month', 'CheckNumber', ]
tmo = tmobile.copy()
tmo[gCols] = tmo[gCols].fillna('xx')
g = tmo.groupby(gCols)

cols = ['Customer', ]
tmo = g[cols].first()

numCols = ['Qty', 'MRC', 'Incoming']
tmo[numCols] = g[numCols].sum()

tmo['in_tmo'] = True

tmo = tmo.reset_index()
sv = ['Customer', 'ActivityType', ]
tmo = tmo.sort_values(cols)
tmo = tmo.round(2)

tmoRoll = tmo
tmo.shape

In [ ]:
tmo.sample(5)

In [ ]:
df = wi.merge(tmo, 
    left_on='PT_Num', 
    right_on='PTNumber',
    how='outer',
    suffixes=('', '_tmo'))
df = df.sort_values('Customer')
df.shape

In [ ]:
#- Supplement act summary
aCols = ['PT_Num', 'Fallout', 'ChangeNote', ]
df = df.merge(actSummary[aCols], on='PT_Num', how='left')
df.shape

In [ ]:
#- fill activity type
df.ActivityType = df.ActivityType.fillna('ACT')
df.ActivityType.value_counts()

In [ ]:
#- attribute reps
idx = df.Rep.isnull()
df.loc[idx, 'Rep'] = df.BAN_tmo.map(bl).fillna('House-Alt')

df[idx].Rep.value_counts()

In [ ]:
#df.Qty.value_counts()

In [ ]:
## Fills
#- WI is better.
df.Customer = df.Customer.fillna(df.Customer_tmo)
df.BAN = df.BAN.fillna(df.BAN_tmo)

#- TMO is better.
df.Qty = df.Qty_tmo.fillna(df.Qty)
df.OrderMRC = df.MRC.fillna(df.OrderMRC)
df.Big = (df.Qty.abs() >= 10)

#- drop redundant
#-- Qty_tmo calculates payment rate
drops = ['BAN_tmo', 'Customer_tmo',]
df = df.drop(columns=drops)

df.Qty.sum()

In [ ]:
#- bools
df = prepBoolCols(df)

In [ ]:
## PaidStatus - easier on the act summary merge
df['PaidStatus'] = pd.NA

idx = df.in_wi & df.in_tmo
df.loc[idx, 'PaidStatus'] = 'Paid Act'

idx = ~df.in_tmo
df.loc[idx, 'PaidStatus'] = 'Unpaid'

#- might want to add 'unmatched?'  I've got unmatched from current check.
idx = ~df.in_wi
df.loc[idx, 'PaidStatus'] = 'Dispute Payment'

idx = df.ActivityType.isin(['DEACT', 'REACT'])
df.loc[idx, 'PaidStatus'] = 'Deact-React'

#- IOT patch
idx = df.IOT.fillna(False)
df.loc[idx, 'PaidStatus'] = 'IOT-Auto'

df.PaidStatus.fillna('xx').value_counts()

In [ ]:
#- Quantity diffs / dispute data
d2 = df.fillna(0)
d2['Qty_wi'] = df.Qty
d2['Qty_diff'] = d2.Qty_wi - d2.Qty_tmo
d2['Qty_ratio'] = d2.Qty_tmo / d2.Qty_wi

d2['MRC_wi'] = d2.OrderMRC
d2['MRC_tmo'] = d2.MRC  
d2['MRC_diff'] = d2.MRC_wi - d2.MRC_tmo

#- dispute data
idx = d2.Qty_ratio < 0.95
idx = idx & ~df.IOT.fillna(False)

disputeData = d2[idx]
disputeData.to_excel(mergedDataFolder+'dispute data.xlsx', index=False)
disputeData.shape

d2.head(2)

In [ ]:
#- Quantity diffs review
ps = ['Paid Act', 'Unpaid']
idx = (d2.Qty >= 5) & d2.PaidStatus.isin(ps)
cols = ['Customer', 'Rep', 'Qty', 'PaidStatus',
        'Qty_wi','Qty_tmo','Qty_diff',
        'MRC_wi','MRC_tmo','MRC_diff',
       ]

diffs = d2.loc[idx, cols].copy()

diffs['Diff'] = (diffs.Qty_diff.abs() > 0) | (diffs.MRC_diff.abs() > 0)
diffs.Diff = diffs.Diff.fillna(False)

diffs.to_excel(mergedDataFolder+'qty diffs.xlsx', index=False)

diffs.head()

### Export

In [ ]:
fn = 'merged data.xlsx'
df.to_excel(mergedDataFolder+fn, index=False)
print('Exported:', fn)

In [ ]:
df.PaidStatus.value_counts()